# Telemetry Pipeline Orchestrator

This notebook orchestrates the Frustration Scoring pipeline. It identifies new raw telemetry files in S3, runs them through the preprocessing and inference pipeline, and marks them as processed to ensure idempotency.

### Workflow:
1. **Scan**: Lists all files in the `raw/` S3 prefix.
2. **Filter**: Skips files that have a correspondig `.done` marker in the `processed_markers/` prefix.
3. **Execute**: Calls `client.py` for each new file.
4. **Schedule**: Once verified, this notebook can be scheduled as a SageMaker Job.

In [ ]:
import boto3
import os
from client import run_synchronized_inference, is_file_processed
from config import RAW_BUCKET, S3_RAW_PREFIX, SAGEMAKER_ENDPOINT_NAME

# Configuration
print(f"Target Bucket: {RAW_BUCKET}")
print(f"Raw Prefix: {S3_RAW_PREFIX}")

In [ ]:
def get_new_files():
    """Lists files in S3_RAW_PREFIX that haven't been processed yet."""
    s3 = boto3.client('s3')
    response = s3.list_objects_v2(Bucket=RAW_BUCKET, Prefix=S3_RAW_PREFIX)
    
    new_files = []
    if 'Contents' in response:
        for obj in response['Contents']:
            key = obj['Key']
            # Skip the prefix itself and non-data files
            if key == S3_RAW_PREFIX or not (key.endswith('.ndjson') or key.endswith('.csv')):
                continue
            
            s3_uri = f"s3://{RAW_BUCKET}/{key}"
            if not is_file_processed(s3_uri):
                new_files.append(s3_uri)
    
    return new_files

files_to_process = get_new_files()
print(f"Found {len(files_to_process)} new files to process.")
for f in files_to_process:
    print(f"  - {f}")

In [ ]:
for s3_uri in files_to_process:
    print(f"\nProcessing: {s3_uri}")
    try:
        run_synchronized_inference(
            raw_data_path=s3_uri,
            endpoint_name=SAGEMAKER_ENDPOINT_NAME,
            local_only=False
        )
    except Exception as e:
        print(f"  [ERROR] Failed to process {s3_uri}: {e}")

## How to Schedule this Notebook

1. **Save** the notebook.
2. Click the **"Create Job"** button (the calendar icon with a plus sign) in the SageMaker Studio toolbar.
3. Set a **Job name** (e.g., `Telemetry-Pipeline-Run`).
4. Choose **"Run on a schedule"**.
5. Select your frequency (e.g., every 5 minutes or once an hour).
6. Click **Create**.

SageMaker will handle the EventBridge rule creation and execution of this notebook for you automatically.